# 03 · SFT a native-schema coding agent

Train a language-only LoRA on validated, replayable trajectories.
The notebook defaults to a two-step plumbing smoke test; the real
run remains gated until the baseline, schema-survival, split, and
replay checks pass.

**Input:** a private dataset from notebook 02.
**Output:** a versioned LoRA adapter, not a merged base model.

## Install the pinned day-zero environment

In [ ]:
import subprocess
import sys
from pathlib import Path

# Day-zero Qwen3.8 support is pinned to commits verified when these notebooks
# were generated. Change all four together after a compatibility smoke test.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
    "transformers": "c49429ed1f8b89749de77c0ec930ef19685c9ae5",
    "trl": "b39c2276567639b93ca5b53658751e0f9c09b92f",
}

INSTALL_KEY = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
FORCE_INSTALL = False

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    packages = [
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"transformers @ git+https://github.com/huggingface/transformers.git@{GIT_REVISIONS['transformers']}",
        f"trl @ git+https://github.com/huggingface/trl.git@{GIT_REVISIONS['trl']}",
        "peft",
        "datasets",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub",
        "sentencepiece",
        "protobuf",
        "pytest",
    ]
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages]
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import json
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
print(f"GPU: {gpu.name} ({gpu_gib:.1f} GiB), capability={torch.cuda.get_device_capability(0)}")
if gpu_gib < 90:
    raise RuntimeError("This suite expects the 96 GB Colab G4 runtime; usable VRAM is below 90 GiB.")

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_gib": round(gpu_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Run configuration

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset, load_dataset
from trl import SFTConfig, SFTTrainer

MODEL_ID = "unsloth/Qwen3.8-27B"
DATASET_ID = f"{HF_USERNAME}/qwen38-code-native-sft-v0"
DATASET_REVISION = "main"  # Replace with an immutable commit SHA for a real run.
OUTPUT_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
MERGED_MODEL_ID = f"{HF_USERNAME}/qwen38-27b-code-accepted-merged"
MAX_SEQ_LENGTH = 4_096       # Use 8_192 only after the 4k memory smoke passes.
MAX_STEPS = 2                # Replace only after the gates pass.
DEMO_MODE = True
RUN_TRAINING = False
PUSH_ADAPTER = False
SAVE_MERGED_BF16 = False
PUSH_MERGED_BF16 = False

if DEMO_MODE and (PUSH_ADAPTER or PUSH_MERGED_BF16 or MAX_STEPS > 2):
    raise RuntimeError("Demo mode is limited to two local smoke steps and cannot be published.")

run_manifest = {
    "stage": "sft",
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_steps": MAX_STEPS,
    "demo_mode": DEMO_MODE,
    "tool_schema_version": "qwen38-six-tools-v1",
    "harness_version": "pilot-local-v1",
    "run_training": RUN_TRAINING,
}
print(json.dumps(run_manifest, indent=2))

## Load the model and discover supported LoRA targets

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    full_finetuning=False,
    token=hf_token,
)

candidate_suffixes = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj", "in_proj", "out_proj",
}
language_linear_names = [
    name for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
    and not any(part in name.lower() for part in ("vision", "visual", "image"))
]
target_modules = sorted({
    name.rsplit(".", 1)[-1]
    for name in language_linear_names
    if name.rsplit(".", 1)[-1] in candidate_suffixes
})
if not target_modules:
    raise RuntimeError("No supported language LoRA targets were discovered; stop and inspect the architecture.")
print("LoRA targets:", target_modules)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=target_modules,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# Qwen3.8 is natively multimodal. This project tunes only language behavior.
for name, parameter in model.named_parameters():
    if any(part in name.lower() for part in ("vision", "visual", "image")):
        parameter.requires_grad_(False)
trainable_vision = [
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad and any(part in name.lower() for part in ("vision", "visual", "image"))
]
assert not trainable_vision, trainable_vision[:20]
model.print_trainable_parameters()

## Load native-schema data and render the exact deployment template

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run a restricted allow-listed command. It is disabled in the pilot.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Fold an initial developer message into system for the HF tokenizer.

    The adapter performs the same mapping in training and deployment. The
    official safetensor tokenizer currently accepts system/user/assistant/tool.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def render_chat(messages: list[dict], *, add_generation_prompt: bool) -> str:
    return tokenizer.apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort="medium",
        preserve_thinking=True,
    )

In [ ]:
def demo_rows():
    return [
        {
            "repo_family": "fixture/clamp",
            "tool_schema_version": "qwen38-six-tools-v1",
            "tool_schema_json": TOOL_SCHEMA_JSON,
            "tools": TOOLS,
            "messages": [
                {"role": "developer", "content": "Fix the bug, run tests, and keep the change minimal."},
                {"role": "user", "content": "clamp() returns values outside its bounds."},
                {"role": "assistant", "content": "", "tool_calls": [{
                    "type": "function",
                    "function": {"name": "read_file", "arguments": {"path": "src/clamp.py"}},
                }]},
                {"role": "tool", "name": "read_file", "content": "def clamp(x, low, high):\n    return x\n"},
                {"role": "assistant", "content": "", "tool_calls": [{
                    "type": "function",
                    "function": {"name": "apply_patch", "arguments": {
                        "patch": "--- a/src/clamp.py\n+++ b/src/clamp.py\n@@ -1,2 +1,2 @@\n def clamp(x, low, high):\n-    return x\n+    return max(low, min(high, x))\n"
                    }},
                }]},
                {"role": "tool", "name": "apply_patch", "content": "Done!"},
                {"role": "assistant", "content": "Implemented the bounded clamp and kept the patch focused."},
            ],
        },
        {
            "repo_family": "fixture/parser",
            "tool_schema_version": "qwen38-six-tools-v1",
            "tool_schema_json": TOOL_SCHEMA_JSON,
            "tools": TOOLS,
            "messages": [
                {"role": "developer", "content": "Investigate first, then make the smallest correct edit."},
                {"role": "user", "content": "Return an empty list for an empty CSV field."},
                {"role": "assistant", "content": "I will inspect the parser and its tests before editing."},
            ],
        },
    ]

USE_DEMO_DATA = DEMO_MODE
if USE_DEMO_DATA:
    raw = Dataset.from_list(demo_rows())
    split = raw.train_test_split(test_size=0.5, seed=3407)
    train_raw, eval_raw = split["train"], split["test"]
    print("Using synthetic plumbing data; this is not a capability run.")
else:
    loaded = load_dataset(DATASET_ID, revision=DATASET_REVISION, token=hf_token)
    missing_splits = {"train", "validation"} - set(loaded)
    if missing_splits:
        raise ValueError(
            f"Dataset is missing required repository-family splits: {sorted(missing_splits)}. "
            "Rebuild it with notebook 02."
        )
    if len(loaded["train"]) == 0 or len(loaded["validation"]) == 0:
        raise ValueError("Both train and validation splits must contain at least one repository family.")
    train_raw = loaded["train"]
    eval_raw = loaded["validation"]

def render_row(row):
    if row.get("tool_schema_version") != "qwen38-six-tools-v1":
        raise ValueError("Dataset schema version differs from this notebook.")
    if row.get("tool_schema_json") != TOOL_SCHEMA_JSON:
        raise ValueError("Dataset canonical tool fingerprint differs from this notebook.")
    if canonical_tool_schema(row.get("tools") or []) != TOOL_SCHEMA_JSON:
        raise ValueError("Dataset tools differ from the deployment tool surface.")
    return {"text": render_chat(row["messages"], add_generation_prompt=False)}

train_dataset = train_raw.map(render_row)
eval_dataset = eval_raw.map(render_row)
print(train_dataset[0]["text"][:4000])

## Build the assistant-only trainer and inspect its labels

In [ ]:
training_args = SFTConfig(
    output_dir=str(RUN_ROOT / "sft"),
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_steps=MAX_STEPS,
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=1,
    save_strategy="steps",
    save_steps=1,
    save_total_limit=2,
    seed=3407,
    report_to="trackio",
    run_name="qwen38-code-sft-smoke" if USE_DEMO_DATA else "qwen38-code-sft",
    push_to_hub=PUSH_ADAPTER,
    hub_model_id=OUTPUT_ADAPTER_ID,
    hub_strategy="every_save",
)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

batch = next(iter(trainer.get_train_dataloader()))
labels = batch["labels"]
assert (labels != -100).any(), "No assistant tokens remain after response masking."
first_trainable = int((labels[0] != -100).nonzero()[0])
assert (labels[0, :first_trainable] == -100).all(), "Prompt/tool context leaked into the first response loss."

supervised_texts = []
for split_dataset in (trainer.train_dataset, trainer.eval_dataset):
    for row in split_dataset:
        supervised_ids = [
            token_id for token_id, label in zip(row["input_ids"], row["labels"])
            if label != -100
        ]
        supervised_texts.append(tokenizer.decode(supervised_ids, skip_special_tokens=False))
joined_supervision = "\n".join(supervised_texts)
assert "Implemented the bounded clamp" in joined_supervision, "Expected final assistant answer is masked."
assert "def clamp(x, low, high)" not in joined_supervision, "Tool observation leaked into the loss."
print({
    "batch_shape": tuple(labels.shape),
    "first_trained_token": first_trainable,
    "supervision_preview": joined_supervision[:2000],
})

## Train, resume, and publish the adapter

In [ ]:
if RUN_TRAINING:
    checkpoints = sorted((RUN_ROOT / "sft").glob("checkpoint-*"))
    torch.cuda.reset_peak_memory_stats()
    start_reserved_gib = torch.cuda.memory_reserved() / 1024**3
    result = trainer.train(resume_from_checkpoint=str(checkpoints[-1]) if checkpoints else None)
    peak_reserved_gib = torch.cuda.max_memory_reserved() / 1024**3
    run_manifest["train_runtime_seconds"] = result.metrics.get("train_runtime")
    run_manifest["peak_reserved_gib"] = round(peak_reserved_gib, 3)
    run_manifest["training_memory_delta_gib"] = round(peak_reserved_gib - start_reserved_gib, 3)
    trainer.save_model(str(RUN_ROOT / "sft" / "final_adapter"))
    tokenizer.save_pretrained(str(RUN_ROOT / "sft" / "final_adapter"))
    (RUN_ROOT / "sft" / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
    if PUSH_ADAPTER:
        trainer.push_to_hub(commit_message="SFT adapter with native six-tool schema")
    print(result.metrics)
    print({
        "peak_reserved_gib": round(peak_reserved_gib, 3),
        "training_memory_delta_gib": round(peak_reserved_gib - start_reserved_gib, 3),
    })
else:
    print("Dry run complete. Set RUN_TRAINING=True only after inspecting labels and memory.")

## Optional merged checkpoint (large and deliberately separate)

In [ ]:
if SAVE_MERGED_BF16:
    if PUSH_MERGED_BF16:
        from huggingface_hub import HfApi

        HfApi(token=hf_token).create_repo(
            repo_id=MERGED_MODEL_ID,
            repo_type="model",
            private=True,
            exist_ok=True,
        )
        model.push_to_hub_merged(
            MERGED_MODEL_ID,
            tokenizer,
            save_method="merged_16bit",
            token=hf_token,
        )
        print(f"Published private merged checkpoint to {MERGED_MODEL_ID}.")
    else:
        merged_dir = RUN_ROOT / "sft" / "merged_16bit"
        model.save_pretrained_merged(
            str(merged_dir), tokenizer, save_method="merged_16bit"
        )
        print(f"Saved merged checkpoint to {merged_dir}.")
else:
    print("Adapter-only is the default durable artifact. Merged BF16 export is disabled.")

## Gate to notebook 04

Training loss is diagnostic, not success. Accept this adapter only
if native tool syntax, held-out patch correctness, non-regression,
and sentinel long-horizon outcomes beat the frozen baseline.
Record the exact adapter commit SHA before preference tuning.